# LAB 02: Augmented Analytics & Venture Capital Financial Valuation
**[🇧🇷 PT-BR]** Automação de Análise Financeira de Startups (Venture Capital)  
**[🇺🇸 EN]** Automated Venture Capital Portfolio Financial Analysis & Capital Allocation  
**Course / Disciplina:** Augmented Analytics & AI-Driven Insights (FIAP MBA)  
**Official Engine / Modelo Oficial:** Gemini 3.8 Flash (`gemini-3.8-flash`) via `google-genai` SDK

---

* 🇧🇷 Este laboratório demonstra como automatizar a análise financeira de portfólios de Venture Capital transformando dados tabulares em memorandos executivos de diretoria (*White-Box Analytics*).
* 🇺🇸 This laboratory demonstrates how to automate Venture Capital portfolio valuation at scale, transforming tabular ledger telemetry into boardroom-ready investment memorandums (*White-Box Analytics*).

### 1. Environment Setup & Authentication / Instalação e Configuração
* 🇧🇷 Instalação do SDK oficial (`google-genai`), configuração de idioma e autenticação com a API Key do Google AI Studio.
* 🇺🇸 Install the official `google-genai` SDK, configure output language, and authenticate with your Google AI Studio API Key.

In [ ]:
# 1. Install official Google GenAI SDK and utilities
!pip install -q -U google-genai tabulate

from google import genai
from google.genai import types
import pandas as pd
import io

# 2. Configure API Key & Output Language
# 🇧🇷 Substitua 'COLE_SUA_API_KEY_AQUI' pela chave do Google AI Studio | 🇺🇸 Replace with your Google AI Studio API Key
API_KEY = 'COLE_SUA_API_KEY_AQUI'
OUTPUT_LANGUAGE = 'Portuguese (Brazil)' # @param ['Portuguese (Brazil)', 'English']
client = genai.Client(api_key=API_KEY)

### 2. Financial Telemetry Ingestion / Carregamento dos Dados Financeiros
* 🇧🇷 Carga resiliente do dataset `Lab 02 - 50_Startups.csv` (P&D, Administração, Marketing, Estado e Lucro Líquido).
* 🇺🇸 Resilient self-healing ingestion of `Lab 02 - 50_Startups.csv` (R&D Spend, Administration, Marketing Spend, State, and Profit).

In [ ]:
# Resilient Self-Healing Dataset Ingestion (50_Startups.csv)
import os
import urllib.request

DATASET_NAME = 'Lab 02 - 50_Startups.csv'
RAW_URL = 'https://raw.githubusercontent.com/rafaelmatsuyama/FIAP-BI-AugmentedAnalytics/main/lab02-financial-analytics/Lab%2002%20-%2050_Startups.csv'

if not os.path.exists(DATASET_NAME):
    print(f"📥 Downloading / Baixando {DATASET_NAME} from official repository...")
    urllib.request.urlretrieve(RAW_URL, DATASET_NAME)

try:
    df = pd.read_csv(DATASET_NAME)
    print('✅ Dataset loaded successfully! / Dataset carregado com sucesso!')
    print(f'Total records / Total de registros: {len(df)}')
    display(df.head())
except Exception as e:
    print(f"❌ Error loading dataset / Erro ao carregar dataset: {e}")

### 3. Strategic Prompt Engineering (VC Persona) / Preparação do Prompt Estratégico
* 🇧🇷 Serialização da amostra em Markdown e calibração da *System Instruction* como Sócio Sênior de Venture Capital.
* 🇺🇸 Serialize the startup sample into Markdown and calibrate the *System Instruction* as a Senior Venture Capital Partner.

In [ ]:
# Serialize a bounded 15-startup sample into Markdown for in-context grounding
dados_amostra = df.head(15).to_markdown()

system_instruction = f"""
Você é um Sócio Sênior de um fundo de Venture Capital (VC) / Senior Venture Capital Partner.
Sua especialidade é identificar eficiência operacional e retorno sobre capital em startups de tecnologia.
Analise os dados financeiros fornecidos e gere um relatório executivo focado em ROI e alocação estratégica.
IMPORTANTE / IMPORTANT: Redija todo o relatório no idioma: {OUTPUT_LANGUAGE}.
"""

prompt_usuario = f"""
Com base na tabela de dados abaixo (amostra de 15 startups), realize as seguintes tarefas analíticas (responda em {OUTPUT_LANGUAGE}):

1. **Análise de Eficiência (R&D Efficiency):** Identifique qual startup apresenta a melhor relação entre investimento em R&D (Pesquisa e Desenvolvimento) e Lucro (Profit).
2. **Diagnóstico de Gastos (G&A Overhead):** Existe alguma correlação visível entre altos gastos com Administração e a queda na margem de lucro?
3. **Recomendação de Board (Series A Thesis):** Escolha a startup que você recomendaria para um aporte de Série A e justifique sua tese com números reais da tabela.

DADOS DAS STARTUPS / STARTUP LEDGER SAMPLE:
{dados_amostra}

DIRETRIZ DE SAÍDA / OUTPUT DIRECTIVE: Estruture a resposta como um 'MEMORANDO DE INVESTIMENTO / VC INVESTMENT MEMORANDUM' executivo em {OUTPUT_LANGUAGE}, com seções claras, tabelas comparativas e justificativas numéricas aprofundadas.
"""

### 4. Gemini 3.8 Flash Inference & Report Generation / Invocação e Geração do Relatório
* 🇧🇷 Execução via `client.models.generate_content` com calibração determinística (`temperature=0.3`, `top_p=0.2`) e `max_output_tokens=10000`.
* 🇺🇸 Invoke `client.models.generate_content` with deterministic sampling (`temperature=0.3`, `top_p=0.2`) and `max_output_tokens=10000`.

In [ ]:
# Execute inference call using the official google-genai SDK
print(f"\n🤖 AI Agent analyzing financial telemetry ({OUTPUT_LANGUAGE})... / Analisando dados financeiros...\n")

response = client.models.generate_content(
    model='gemini-3.8-flash',
    contents=prompt_usuario,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.3,
        top_p=0.2,
        top_k=40,
        max_output_tokens=10000
    )
)

print("-" * 80)
print("📊 VC BOARD INVESTMENT MEMORANDUM / RELATÓRIO ESTRATÉGICO DE INVESTIMENTO")
print("-" * 80)
print(response.text)